<a href="https://colab.research.google.com/github/sanagahoi/log-classifier/blob/main/Log_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python --version

Python 3.11.3


In [2]:
import pandas as pd

df = pd.read_csv("synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   timestamp     2410 non-null   object
 1   source        2410 non-null   object
 2   log_message   2410 non-null   object
 3   target_label  2410 non-null   object
 4   complexity    2410 non-null   object
dtypes: object(5)
memory usage: 94.3+ KB


In [4]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [5]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [7]:
!pip install sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 4.0.2 requires pydantic>=2.0, but you have pydantic 1.10.13 which is incompatible.
gradio 4.0.2 requires websockets<12.0,>=10.0, but you have websockets 16.0 which is incompatible.
gradio-client 0.7.0 requires websockets<12.0,>=10.0, but you have websockets 16.0 which is incompatible.



  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 7.0 MB/s  0:00:00
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   --------- ------------------------------ 2.6/10.6 MB 11.6 MB/s eta 0:00:01
   ---------------- ----------------------- 4.5/10.6 MB 10.8 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.6 MB 10.3 MB/s eta 0:00:01
   ------------------------------ --------- 8.1/10.6 MB 9.3 MB/s eta 0:00:01
   ----------------------------------- ---- 9.4/10.6 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.6 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 7.5 MB/s  0:00:01
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 663.6/663.6 kB 3.7 MB/s  0:00:00
   ---------------------------

In [8]:
# all different patterns to use regex e.g. kmeans clustering and DBScan
# here we're using DBScan and SentenceTransformer

from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import DBSCAN

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['log_message'].tolist())

embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

array([[-0.10293962,  0.03354593, -0.02202606, ...,  0.00457791,
        -0.04259717,  0.00322621],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296],
       ...,
       [-0.04022264,  0.0422436 , -0.06610421, ...,  0.0236367 ,
        -0.00530877,  0.0204446 ],
       [-0.03603452,  0.01960893,  0.10052755, ...,  0.03668106,
        -0.02487851, -0.00578846],
       [ 0.01457431,  0.04911832, -0.00301354, ...,  0.01029741,
        -0.00068493,  0.00708859]], dtype=float32)

In [9]:
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
dbscan.fit(embeddings)

df['cluster'] = dbscan.labels_
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [10]:
 # check some values - by custom values of hyper parameters we get cluster 1 for similar msgs

df[df.cluster==1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,bert,1
477,12/2/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,bert,1
570,11/7/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,bert,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,bert,1


In [11]:
cluster_counts = df['cluster'].value_counts()
large_clusters = cluster_counts[cluster_counts>10].index

for cluster in large_clusters:
    print(f"Cluster {cluster}:")
    print(df[df['cluster'] == cluster]['log_message'].head().to_string(index= False))
    print()

Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...

Cluster 11:
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.

Cluster 13:
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.

Cluster 7:
Multiple bad login attempts detected on user 85...
Multiple login failures occurred on user 9052 a...
  User 

In [12]:
# capture each message log
import re

def classify_log_message(log_message):
  regex_patterns ={
      r"User User\d+ logged (in|out)": "User Action",
      r"Backup (started|ended) at .*": "System Notification",
      r"Backup completed successfully.": "System Notification",
      r"Backup failed with error: .*": "System Notification",
      r"System updated to version .*": "System Notification",
      r"Disk cleanup completed successfully.": "System Notification",
      r"Disk cleanup failed with error: .*": "System Notification",
      r"System reboot initiated.": "System Notification",
      r"System shutdown initiated.": "System Notification",
      r"System update initiated.": "System Notification",
      r"Account with ID .* created by .*" : "User Action",
      r"Account with ID .* deleted by .*" : "User Action",
      r"Account with ID .* updated by .*" : "User Action",
      r"Account with ID .* locked by .*" : "User Action",
      r"Account with ID .* unlocked by .*" : "User Action",
      r"Account with ID .* disabled by .*" : "User Action",
      r"Account with ID .* enabled by .*" : "User Action",
      r"Account with ID .* added to group .* by .*" : "User Action",
      r"Account with ID .* removed from group .* by .*" : "User Action",
      r"Account with ID .* added to role .* by .*" : "User Action",
      r"Account with ID .* removed from role .* by .*" : "User Action",
      r"Account with ID .* added to policy .* by .*" : "User Action",
      r"Account with ID .* removed from policy .* by .*" : "User Action"
  }

  for pattern, label in regex_patterns.items():
    if re.search(pattern, log_message, re.IGNORECASE):
      return label
  return None

In [13]:
classify_log_message("System update initiated.")

'System Notification'

In [14]:
# apply this function to log_message column
df['regex_label'] = df['log_message'].apply(classify_log_message)
df.head()

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None


In [15]:
df_non_regex = df[df.regex_label.isna()].copy()
df_non_regex.head()

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None


In [16]:
df_non_regex.shape

(1963, 7)

In [17]:
df_non_regex['target_label'].value_counts()

HTTP Status            1017
Security Alert          371
Error                   177
Resource Usage          177
Critical Error          161
System Notification      53
Workflow Error            4
Deprecation Warning       3
Name: target_label, dtype: int64

In [18]:
# for less data in workflow error and deprecation warning, we can use LLM butfirst well create embedding
df_non_legacy = df_non_regex[df_non_regex.source != 'LegacyCRM']

In [19]:
df_non_legacy.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [20]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())
filtered_embeddings

array([[-0.10293962,  0.03354593, -0.02202606, ...,  0.00457791,
        -0.04259717,  0.00322621],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296],
       ...,
       [-0.04022264,  0.0422436 , -0.06610421, ...,  0.0236367 ,
        -0.00530877,  0.0204446 ],
       [-0.03603452,  0.01960893,  0.10052755, ...,  0.03668106,
        -0.02487851, -0.00578846],
       [ 0.01457431,  0.04911832, -0.00301354, ...,  0.01029741,
        -0.00068493,  0.00708859]], dtype=float32)

In [21]:
# logistic regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X = filtered_embeddings
y = df_non_legacy['target_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

                     precision    recall  f1-score   support

     Critical Error       0.91      1.00      0.95        29
              Error       1.00      0.91      0.95        32
        HTTP Status       1.00      1.00      1.00       217
     Resource Usage       1.00      1.00      1.00        28
     Security Alert       1.00      1.00      1.00        77
System Notification       1.00      1.00      1.00         9

           accuracy                           0.99       392
          macro avg       0.98      0.98      0.98       392
       weighted avg       0.99      0.99      0.99       392



In [23]:
import joblib
joblib.dump(model, 'model.joblib')

['model.joblib']